# Chromagram review

Look at a sample recording, decide which spans hold which harmony, and export
those decisions to `tests/data/recordings.json` for `tests/test_recording_samples.py`
to assert against.

Two questions per window, matching the two tests:

1. **Do the expected notes dominate the 12-bin chromagram?**
2. **Does their power sit on the semitone center, or has the take drifted out of tune?**

The recordings are not in the repository. Point `MANGOMUSIC_SAMPLES_DIR` at them,
or keep them in `Recording_samples/` beside the repo, which is the default.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mangomusic.audio import load_audio
from mangomusic.chroma import (
    BINS_PER_SEMITONE,
    PITCH_CLASS_NAMES,
    compute_chroma,
    compute_semitone_chroma,
)
from mangomusic.recordings import (
    RecordingManifest,
    RecordingWindow,
    recording_path,
    samples_dir,
)

SAMPLE_RATE_HZ = 22_050
MANIFEST_PATH = Path.cwd().parent / "tests" / "data" / "recordings.json"

# Slot 1 of the reference categorical palette, with a neutral for context marks.
ACCENT = "#2a78d6"
CONTEXT = "#9a9a95"
INK = "#0b0b0b"
MUTED = "#52514e"

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": MUTED,
        "axes.labelcolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "text.color": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "grid.color": "#e4e4e0",
        "grid.linewidth": 0.8,
        "figure.dpi": 110,
    }
)

print("samples dir:", samples_dir())

## 1. Choose a recording

`sorted(...)` lists what is actually present, so a renamed file shows up here
rather than failing further down.

In [ ]:
directory = samples_dir()
if directory is None:
    raise SystemExit("No samples directory. Set MANGOMUSIC_SAMPLES_DIR.")

for path in sorted(directory.glob("*.wav")):
    print(path.name)

In [ ]:
RECORDING = "guitar_09_both_triads.wav"

path = recording_path(RECORDING)
if path is None:
    raise SystemExit(f"Not found: {RECORDING}")

samples, sample_rate_hz = load_audio(path, SAMPLE_RATE_HZ)
chroma, frame_times_seconds = compute_chroma(samples, sample_rate_hz)
print(f"{RECORDING}: {frame_times_seconds[-1]:.1f}s, {chroma.shape[1]} frames")

## 2. The chromagram over time

Magnitude gets a single-hue sequential ramp, light to dark. Use this to find
where the harmony actually sounds — these recordings open and close with several
seconds of noise floor, which shows up as a flat wash with no bright rows, and
must be excluded from any window you annotate.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.6))

ax.imshow(
    chroma,
    aspect="auto",
    origin="lower",
    cmap="Blues",
    interpolation="nearest",
    extent=(0.0, float(frame_times_seconds[-1]), -0.5, 11.5),
)
ax.set_yticks(range(12), PITCH_CLASS_NAMES)
ax.set_xlabel("time (seconds)")
ax.set_title(f"{RECORDING} — pitch-class energy", loc="left")
ax.grid(False)
fig.colorbar(ax.images[0], ax=ax, label="normalized energy", pad=0.01)
plt.show()

## 3. Pick a window

Set the span you want to annotate and what you played in it — either a `CHORD`
symbol such as `"C:maj"`, or `NOTES` such as `("C",)` for a single sustained note.
Set exactly one; the other stays `None`.

In [ ]:
WINDOW_START_SECONDS = 6.0
WINDOW_STOP_SECONDS = 8.5
CHORD: str | None = "C:maj"
NOTES: tuple[str, ...] | None = None

window = RecordingWindow(
    start_seconds=WINDOW_START_SECONDS,
    stop_seconds=WINDOW_STOP_SECONDS,
    chord=CHORD,
    notes=NOTES,
)
expected = window.expected_pitch_classes

window_samples, _ = load_audio(
    path,
    SAMPLE_RATE_HZ,
    start_time_seconds=window.start_seconds,
    stop_time_seconds=window.stop_seconds,
)
window_chroma, _ = compute_chroma(window_samples, SAMPLE_RATE_HZ)
window_semitone, _ = compute_semitone_chroma(window_samples, SAMPLE_RATE_HZ)

summary = window_chroma.astype(np.float64).mean(axis=1)
semitone_summary = window_semitone.astype(np.float64).mean(axis=1)

print("expecting:", [PITCH_CLASS_NAMES[i] for i in expected])

## 4. Do the expected notes dominate? (12-bin)

The test ranks the pitch classes and asserts the top *n* are exactly the expected
ones, with the weakest expected bin clearing the strongest unexpected bin by a
margin. Both are drawn below — the dashed rule is the margin the last expected
bin has to beat.

In [ ]:
PITCH_CLASS_MARGIN = 1.4

ranked = np.argsort(summary)[::-1]
strongest_other = float(summary[ranked[len(expected)]])
threshold = PITCH_CLASS_MARGIN * strongest_other

fig, ax = plt.subplots(figsize=(9, 3.4))
colors = [ACCENT if i in expected else CONTEXT for i in range(12)]
ax.bar(range(12), summary, color=colors, width=0.68)
ax.axhline(threshold, color=MUTED, linestyle="--", linewidth=1.2)
ax.annotate(
    f"{PITCH_CLASS_MARGIN:g}x strongest other",
    xy=(11.4, threshold),
    xytext=(0, 6),
    textcoords="offset points",
    ha="right",
    fontsize=9,
    color=MUTED,
)
for i in expected:
    ax.annotate(
        PITCH_CLASS_NAMES[i],
        xy=(i, summary[i]),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        color=INK,
    )
ax.set_xticks(range(12), PITCH_CLASS_NAMES)
ax.set_ylabel("mean energy")
ax.set_title(
    f"{RECORDING}  {window.start_seconds:g}-{window.stop_seconds:g}s", loc="left"
)
ax.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color=ACCENT, label="expected"),
        plt.Rectangle((0, 0), 1, 1, color=CONTEXT, label="other"),
    ],
    frameon=False,
    loc="upper right",
)
ax.grid(axis="y", alpha=0.6)
ax.set_axisbelow(True)
plt.show()

got = tuple(sorted(int(i) for i in ranked[: len(expected)]))
print("expected:", [PITCH_CLASS_NAMES[i] for i in expected])
print("strongest:", [PITCH_CLASS_NAMES[i] for i in got])
print("match:", got == expected)

## 5. Is the power on the semitone center? (36-bin)

Each expected pitch class occupies three bins: roughly 33 cents flat, centered,
and roughly 33 cents sharp. An in-tune note puts its energy in the middle bar.
A flat take shifts it left — which the 12-bin view above cannot show, because
there the same shift just leaks into a neighbouring pitch class and looks
identical to playing a different note.

In [ ]:
CENTERED_ENERGY_RATIO = 1.25
OFFSET_LABELS = ("-33c", "centered", "+33c")

fig, axes = plt.subplots(
    1, len(expected), figsize=(3.1 * len(expected), 3.4), squeeze=False, sharey=True
)
for ax, pitch_class in zip(axes[0], expected, strict=True):
    start = BINS_PER_SEMITONE * pitch_class
    trio = semitone_summary[start : start + BINS_PER_SEMITONE]
    ratio = float(trio[1] / max(trio[0], trio[2]))
    in_tune = ratio > CENTERED_ENERGY_RATIO
    ax.bar(
        range(3),
        trio,
        color=[CONTEXT, ACCENT if in_tune else "#eb6834", CONTEXT],
        width=0.62,
    )
    ax.set_xticks(range(3), OFFSET_LABELS, fontsize=8)
    ax.set_title(
        f"{PITCH_CLASS_NAMES[pitch_class]} — {ratio:.2f}x"
        f"\n{'in tune' if in_tune else 'OFF CENTER'}",
        loc="left",
        fontsize=10,
    )
    ax.grid(axis="y", alpha=0.6)
    ax.set_axisbelow(True)
axes[0][0].set_ylabel("mean energy")
fig.suptitle(
    f"centered vs neighbouring bins (threshold {CENTERED_ENERGY_RATIO:g}x)",
    x=0.01,
    ha="left",
)
fig.tight_layout()
plt.show()

## 6. Export to the manifest

Collect the windows you settled on and write them out. This overwrites
`tests/data/recordings.json`, so start from the committed manifest and edit it
rather than rebuilding from scratch, unless you mean to replace everything.

Set `expect_in_tune=False` for a window you know is off — the test then asserts
the check *catches* it, which is what makes `guitar_11_out_of_tune.wav` a passing
test rather than a failing one.

In [ ]:
manifest = RecordingManifest.from_path(MANIFEST_PATH)

# Replace this recording's windows with the one reviewed above. Append instead
# of assigning to keep the windows already recorded for this file.
manifest.recordings[RECORDING] = [
    RecordingWindow(
        start_seconds=window.start_seconds,
        stop_seconds=window.stop_seconds,
        chord=CHORD,
        notes=NOTES,
        expect_in_tune=True,
    )
]

for name, windows in sorted(manifest.recordings.items()):
    for entry in windows:
        label = entry.chord or ",".join(entry.notes or ())
        print(
            f"{name:32s} {entry.start_seconds:5.1f}-{entry.stop_seconds:5.1f}  {label}"
        )

In [ ]:
WRITE = False  # flip to True once the listing above is what you want

if WRITE:
    MANIFEST_PATH.write_text(
        json.dumps(json.loads(manifest.model_dump_json()), indent=2) + "\n"
    )
    print(f"wrote {MANIFEST_PATH}")
else:
    print("nothing written; set WRITE = True to save")

Then run the tests against the recordings:

```sh
uv run pytest tests/test_recording_samples.py -v
```